# Default Assignment Baseline

This notebook builds the first classical baseline for the Nestle DOM challenge: the "as-is" world where nothing gets diverted. Every order just sits at whatever DC SAP already assigned it to (its default DC), and if that DC can't fully cover it, the shortfall gets penalized instead of being fixed by sending the order somewhere else.

The point of this baseline isn't to be smart -- it's to be the honest starting point that everything else (the greedy heuristic later, the QUBO/optimization model later) gets measured against. To see if our fancier methods can or can't beat this.

We'll work off the real POC data pack (anonymized) and the objective function from the September 2022 `DOM Equations.docx`:

$$\text{Objective} = \text{Revenue} - \text{Penalty} - \text{Shipping Cost}$$

By the end we'll have a per-order table (Order, DC, Objective, Fill rate %, Sales, Penalty, Shipping cost -- the standard columns for reporting this kind of result) plus a summary row, and a quick sanity check against the original POC's own output file.

## Files you need to upload

Before running anything below, upload these files from the shared data pack (the `DOM-data` folder). In Colab the easiest way is `files.upload()` in the next cell -- it'll pop up a picker, just select all five at once:

- `input_order data.csv` (the order book -- note the space in the filename, that's intentional, not a typo)
- `input_capacity_planning.csv` (daily inventory by DC/SKU)
- `input_shipping_cost_data.csv` (lane rate card)
- `input_dock_capacity.csv` (used to reproduce which orders count as "focus orders")
- `Output_order_level_data.csv` (the original POC's own output, only used at the end to sanity-check our numbers -- not used anywhere in the baseline calculation itself)

You don't need `input_throughput_capacity.csv` or `output_order_sku_level_data.csv` for this particular notebook.

In [1]:
# standard imports, nothing fancy needed for this baseline
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

In [3]:
# run this cell and pick all 5 files from the list above when the upload box appears
from google.colab import files
uploaded = files.upload()

Saving input_capacity_planning.csv to input_capacity_planning.csv
Saving input_dock_capacity.csv to input_dock_capacity.csv
Saving input_order data.csv to input_order data.csv
Saving input_shipping_cost_data.csv to input_shipping_cost_data.csv


## Step 1 -- load the order book

`input_order data.csv` is one row per order-SKU line (25,193 rows for 1,109 distinct orders). It's already pre-filtered by SAP (the company) to open orders that qualify as full truckloads, so we don't have to redo that filtering ourselves -- it's baked in.

In [4]:
orders = pd.read_csv('input_order data.csv', na_values=['null'])

print(orders.shape)
print('distinct orders:', orders['Group_Flag'].nunique())
orders.head(2)

(25193, 39)
distinct orders: 1109


,Group_Flag,Plant,MaterialNumber,transportationplanningdate,CustomerGroup5Description,IsTopCust,OpeningStock,RequestedDeliveryDate,DeliveryNoteFlag,IsInvAvail,LoadNumber,DeliveryPriority,ProductPlanningUnitOfMeasure,ProductPlanningUnitsPerCase,ProductPlanningUnitsPerPallet,OrderedQty_converted,OrderedWeight,OrderedVolume,Order_SKU_Revenue,ZipCode,ProductCasesPerPallet,CalculatedFootprints,IsCOF<100,calculated_ordered_weight,IsFTL,IsMultiplePlant,IsMultiplePGI,IsMultipleRDD,Measure,FillRateThreshold,Penaltyforpotentialcuts,MaximumPenalty,FixedPenalty,FixedPenaltyPerSKU,MinimumPenalty,Additionalcomments,OnTimePercentage,OnTimeFixed,Unnamed: 38
0,5484913123,5083,12260382,6/26/24,NaN,N,33814.0,6/27/24,N,Y,U600105191,99,CS,1.0,120.0,24,386.435,10.032,1150,431,120.0,7,Y,39463.21,Y,N,N,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5484913123,5083,9516458,6/26/24,NaN,N,6546.0,6/27/24,N,Y,U600105191,99,CS,1.0,80.0,10,90.078,7.360,478,431,80.0,7,Y,39463.21,Y,N,N,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2 -- figure out which orders even need a decision

Not every order is a candidate for reassignment. Per the 2022 business logic, an order only becomes a "focus order" if its default DC can't fully cover it. So first we check, for every SKU line, whether inventory was available (`IsInvAvail`), and roll that up: an order only counts as "sufficient" if *every one* of its lines is covered.

$$\text{order sufficient} = \bigwedge_{s \in \text{order}} \big(\text{IsInvAvail}_{s} = Y\big)$$

This is a per-order boolean, not per line.

In [5]:
# one row per order: default plant, ship date, and whether ALL its lines had inventory
order_level = orders.groupby('Group_Flag').agg(
    Plant=('Plant', 'first'),
    ZipCode=('ZipCode', 'first'),
    ship_date_raw=('transportationplanningdate', 'first'),
    all_lines_available=('IsInvAvail', lambda s: (s == 'Y').all())
).reset_index()

sufficient_ids   = order_level.loc[order_level['all_lines_available'], 'Group_Flag']
insufficient_ids = order_level.loc[~order_level['all_lines_available'], 'Group_Flag']

print('sufficient at default DC:', len(sufficient_ids))
print('insufficient at default DC:', len(insufficient_ids))

sufficient at default DC: 662
insufficient at default DC: 447


## Step 3 -- the dock/throughput addition

Even an order with "sufficient" inventory can still get pulled into the focus list if its DC has no dock capacity left on the day it's supposed to ship -- the 2022 business rules call this the throughput/dock addition. The data pack doesn't give us a clean daily throughput ceiling to check against directly (`input_throughput_capacity.csv` only reports how much capacity got *used*, never the maximum).

Input_throughput_capacity.csv only has util_case_picks and util_pallets — that's "how much got used." There's no remaining column and no capacity column anywhere in this file.

So as a stand-in we use `input_dock_capacity.csv`: if `Dock_Remaining = 0` for a plant on a given day, we treat that day as fully booked.

$$\text{flagged} = \{o \in \text{sufficient} : (\text{plant}_o,\ \text{ship date}_o) \in \{(p,d) : \text{dock remaining}_{p,d} = 0\}\}$$

Worth flagging up front: this is a proxy, not the real rule -- dock appointment slots and case-pick/pallet-pick throughput are related but different physical bottlenecks, and the data pack doesn't expose the real throughput ceiling anywhere. So treat the count that comes out of this step as "how many orders hit a dock scheduling constraint," not as a precise reproduction of the original 3-consecutive-day throughput check.

In [6]:
dock = pd.read_csv('input_dock_capacity.csv')

# the set of (plant, date) combos that are fully booked
fully_booked = set(
    zip(dock.loc[dock['Dock_Remaining'] == 0, 'Plant'],
        dock.loc[dock['Dock_Remaining'] == 0, 'Date'])
)

print('fully-booked plant-day combos:', len(fully_booked))

fully-booked plant-day combos: 43


In [7]:
# check each "sufficient" order against the fully-booked set
sufficient_orders = order_level[order_level['Group_Flag'].isin(sufficient_ids)]

is_flagged = sufficient_orders.apply(
    lambda r: (r['Plant'], r['ship_date_raw']) in fully_booked, axis=1
)
flagged_ids = sufficient_orders.loc[is_flagged, 'Group_Flag']

print('sufficient orders pulled in by dock constraint:', len(flagged_ids))

sufficient orders pulled in by dock constraint: 25


## Step 4 -- the final focus-order list

Focus orders = insufficient orders + the dock-flagged additions. Everyone else already gets fulfilled cleanly at their default DC and isn't part of this baseline comparison at all -- there's no reassignment decision to make for them, so including them would just pad the metrics with 100%-fill, zero-penalty rows and hide what the baseline is actually testing.

That gives 447 + 25 = 472 focus orders. Worth being upfront that this precise number is somewhat proxy-dependent rather than an exact fact about the business, for two reasons:

1. `IsInvAvail` is a static per-line snapshot -- it tells us whether inventory existed at the time the extract was taken, not whether that inventory would survive every other order competing for the same SKU/DC/day. Later in this notebook we compare our fill-rate numbers against the original POC's actual allocation outcome, and a handful of orders that look "sufficient" here turn out to have come up short once everything was really allocated -- because other orders, especially priority ones, drew on the same shared stock first. So "insufficient" as computed here is closer to a lower bound than an exact count.
2. The 2022 business rules also flag a DC as constrained if it lacks *throughput* capacity (case-pick/pallet-pick volume) for several consecutive days -- not just dock appointments. This data pack only gives us daily dock-appointment capacity, not a throughput ceiling, so the dock-based proxy above is standing in for a related-but-not-identical constraint. It's very likely the real throughput ceiling (if we had it) would flag a different, probably smaller, set of orders.

None of this means 472 is "the wrong number" -- it means it's the one produced by the most transparent, fully-reproducible method available given what's actually in the files, rather than one that requires guessing at an internal threshold that simply isn't present in this data pack. That's the number this baseline -- and everything built on top of it -- uses going forward.

In [8]:
focus_order_ids = pd.concat([insufficient_ids, flagged_ids]).unique()

print('focus orders:', len(focus_order_ids))
print('clean orders (excluded from this baseline):', order_level['Group_Flag'].nunique() - len(focus_order_ids))

focus orders: 472
clean orders (excluded from this baseline): 637


## Step 5 -- how many cases actually get filled at the default DC

Now the real baseline math starts. For every order-SKU line, we need to know how many cases the default DC can actually hand over, using the inventory ledger in `input_capacity_planning.csv`. The equations doc defines this as a simple cap:

$$\text{Cases}_{filled} = \min\big(\text{OrderedQty}, \text{Available inventory}\big)$$

We join on distribution center, SKU, and the planned ship date.

In [9]:
capacity = pd.read_csv('input_capacity_planning.csv')

capacity.shape

(377504, 23)

In [10]:
# dates are in different formats in the two files, so line them up first
orders['ship_date']   = pd.to_datetime(orders['transportationplanningdate'], format='%m/%d/%y')
capacity['cal_date']  = pd.to_datetime(capacity['DATE'])

In [11]:
# join each order-SKU line to the inventory available at its default DC on its ship date
lines = orders.merge(
    capacity[['LocationID', 'MaterialID', 'cal_date', 'Available_inventory']],
    left_on=['Plant', 'MaterialNumber', 'ship_date'],
    right_on=['LocationID', 'MaterialID', 'cal_date'],
    how='left'
)

# a handful of lines (14, out of 25193) don't find a matching inventory row at all --
# treating "no record" as "nothing available" rather than dropping the line
lines['Available_inventory'] = lines['Available_inventory'].fillna(0)

print('lines with no inventory record:', lines['Available_inventory'].isna().sum())

lines with no inventory record: 0


In [12]:
# a small number of Available_inventory values in the raw data are slightly negative
# (looks like a rounding/timing artifact upstream) -- can't fill negative cases, so floor at 0
lines['Available_inventory'] = lines['Available_inventory'].clip(lower=0)

# this is the actual constraint from the equations doc: Cases_filled = min(ordered, available)
lines['cases_filled'] = np.minimum(lines['OrderedQty_converted'], lines['Available_inventory'])

## Step 6 -- revenue on what actually shipped, not what was ordered

This is the one place I want to be careful, because it's easy to get wrong just by eyeballing a slide: revenue only counts cases that actually go out the door. An order that gets zero cases filled earns zero revenue in this baseline, even though `Order_SKU_Revenue` in the raw file is priced on the full ordered quantity.

$$\text{Revenue}_{line} = \frac{\text{Cases}_{filled}}{\text{OrderedQty}} \times \text{Order SKU revenue}$$

In [13]:
# guard against divide-by-zero for any line that was ordered as 0 units (shouldn't happen, but just in case)
lines['fill_fraction'] = np.where(
    lines['OrderedQty_converted'] > 0,
    lines['cases_filled'] / lines['OrderedQty_converted'],
    0
)

lines['revenue_earned'] = lines['fill_fraction'] * lines['Order_SKU_Revenue']

## Step 7 -- roll everything up to order level

Everything so far has been at the SKU-line grain. Time to collapse to one row per order -- summing ordered quantity, cases filled, and revenue across all of an order's SKU lines, and carrying along the fields we'll need later (plant, zip, and the penalty schedule).

In [14]:
order_metrics = lines.groupby('Group_Flag').agg(
    Plant=('Plant', 'first'),
    ZipCode=('ZipCode', 'first'),
    ordered_qty=('OrderedQty_converted', 'sum'),
    cases_filled=('cases_filled', 'sum'),
    revenue=('revenue_earned', 'sum'),
    # penalty schedule fields are the same across every line of a given order, so 'first' is safe here
    fill_rate_threshold=('FillRateThreshold', 'first'),
    penalty_rate=('Penaltyforpotentialcuts', 'first')
).reset_index()

order_metrics.head(3)

,Group_Flag,Plant,ZipCode,ordered_qty,cases_filled,revenue,fill_rate_threshold,penalty_rate
0,5478570251,5620,906,6930,6930.0,75503.0,NaN,NaN
1,5478664331,5420,19,800,800.0,88424.0,NaN,NaN
2,5480513396,5620,953,52,52.0,356600.0,1.0,0.0


## Step 8 -- fill rate

Straightforward once we have order-level totals -- total cases filled over total cases ordered, case-weighted (not an average of each line's percentage, so one badly-short line on a big SKU doesn't get washed out by five small fully-stocked ones).

$$\text{fillRate}_o = \frac{\sum \text{Cases}_{filled}}{\sum \text{OrderedQty}}$$

In [15]:
order_metrics['fill_rate'] = order_metrics['cases_filled'] / order_metrics['ordered_qty']

## Step 9 -- shipping cost

Every order ships from its default DC to its own delivery zip, and `input_shipping_cost_data.csv` gives a lane rate for every (plant, zip) combination. Each order only has one plant and one zip (checked this beforehand -- no order spans multiple plants or multiple zips in this file), so it's a clean one-to-one lookup, no aggregation needed.

$$\text{ShippingCost}_o = \text{shipping cost at}\ (Plant_o,\ ZipCode_o)$$

In [16]:
shipping = pd.read_csv('input_shipping_cost_data.csv')

order_metrics = order_metrics.merge(
    shipping[['Plant', 'TargetZip', 'Shipping_Cost']],
    left_on=['Plant', 'ZipCode'],
    right_on=['Plant', 'TargetZip'],
    how='left'
)

print('orders with no shipping rate found:', order_metrics['Shipping_Cost'].isna().sum())

orders with no shipping rate found: 0


## Step 10 -- penalty

This is the field where the raw data is genuinely incomplete, so it's worth spelling out exactly what the equations doc says and where we stop.

The 2022 logic charges a penalty only once fill rate drops **below a customer-specific threshold** (`FillRateThreshold`), and only at a **customer-specific rate** (`Penaltyforpotentialcuts`) applied to the *unfilled* revenue:

$$
\text{Penalty}_o =
\begin{cases}
(\text{ordered qty}_o - \text{cases filled}_o) \times \text{avg price}_o \times \text{penalty rate}_o
 & \text{if fill rate}_o < \text{threshold}_o \\
0 & \text{if fill rate}_o \ge \text{threshold}_o \text{, or no schedule exists for this order}
\end{cases}
$$

About 1 in 5 order-SKU lines have `FillRateThreshold`/`Penaltyforpotentialcuts` as null -- meaning that customer simply has no penalty schedule on file for this run. I checked this against the original POC's own output before trusting it: an order with zero fill rate and no threshold populated shows `PenaltyIfNotDiverted = 0` in `Output_order_level_data.csv` too, so "no schedule -> no penalty" isn't a guess, it's what the real output does.

In [17]:
# average price per unit, needed for the penalty formula (revenue on the full order / full ordered qty)
full_line_revenue = lines.groupby('Group_Flag')['Order_SKU_Revenue'].sum()

order_metrics['avg_price'] = np.where(
    order_metrics['ordered_qty'] > 0,
    full_line_revenue.reindex(order_metrics['Group_Flag']).values / order_metrics['ordered_qty'],
    0
)

In [18]:
# penalty only activates if a threshold exists AND fill rate is actually below it
has_schedule = order_metrics['fill_rate_threshold'].notna() & order_metrics['penalty_rate'].notna()
below_threshold = order_metrics['fill_rate'] < order_metrics['fill_rate_threshold']

penalty_applies = has_schedule & below_threshold

order_metrics['penalty'] = np.where(
    penalty_applies,
    (order_metrics['ordered_qty'] - order_metrics['cases_filled'])
        * order_metrics['avg_price'] * order_metrics['penalty_rate'],
    0.0
)

print('orders with a penalty charged:', (order_metrics['penalty'] > 0).sum())

orders with a penalty charged: 188


## Step 11 -- the objective function

Now that we have revenue, penalty, and shipping cost per order, the objective is just the formula from the equations doc:

$$\text{Objective}_o = \text{Revenue}_o - \text{Penalty}_o - \text{ShippingCost}_o$$

And since this is the default-assignment baseline, every order stays exactly where it started -- there is no reassignment logic anywhere above, so the "# reassigned" count for this baseline is 0 by construction, not something we need to calculate.

In [19]:
order_metrics['objective'] = (
    order_metrics['revenue'] - order_metrics['penalty'] - order_metrics['Shipping_Cost']
)

order_metrics[['Group_Flag','Plant','fill_rate','revenue','penalty','Shipping_Cost','objective']].head(3)

,Group_Flag,Plant,fill_rate,revenue,penalty,Shipping_Cost,objective
0,5478570251,5620,1.0,75503.0,0.0,387,75116.0
1,5478664331,5420,1.0,88424.0,0.0,1151,87273.0
2,5480513396,5620,1.0,356600.0,0.0,1165,355435.0


## Step 12 -- narrow down to the focus orders

Time to bring in the focus-order list from Step 4. Everything above was computed for all 1,109 orders (which is useful, and we glanced at it above), but the actual baseline table we report should only cover the focus orders -- those are the only ones where a reassignment decision is even on the table.

In [20]:
focus_metrics = order_metrics[order_metrics['Group_Flag'].isin(focus_order_ids)].copy()

print('focus orders in final table:', len(focus_metrics))

focus orders in final table: 472


## Step 13 -- the per-order table

Building this with the standard columns for reporting this kind of order-level result: Order, DC, Objective, Fill rate %, Sales ($), Penalty ($), Shipping cost ($).

In [21]:
baseline_table = focus_metrics[[
    'Group_Flag', 'Plant', 'objective', 'fill_rate', 'revenue', 'penalty', 'Shipping_Cost'
]].rename(columns={
    'Group_Flag': 'Order',
    'Plant': 'DC',
    'objective': 'Obj. function [$]',
    'fill_rate': 'Fillrate [%]',
    'revenue': 'Sales [$]',
    'penalty': 'Penalty [$]',
    'Shipping_Cost': 'Shipping cost [$]'
})

# fill rate as a whole-number percentage, same style as the reference table
baseline_table['Fillrate [%]'] = (baseline_table['Fillrate [%]'] * 100).round(0).astype(int)

for col in ['Obj. function [$]', 'Sales [$]', 'Penalty [$]', 'Shipping cost [$]']:
    baseline_table[col] = baseline_table[col].round(0).astype(int)

baseline_table.head(10)

,Order,DC,Obj. function [$],Fillrate [%],Sales [$],Penalty [$],Shipping cost [$]
3,5481093900,5083,83092,100,86911,0,3819
26,5483184998,5420,100175,98,100700,0,525
29,5483287940,5410,73538,100,75610,0,2072
30,5483351229,5420,111850,98,112919,0,1069
35,5483875303,5620,44766,69,45101,0,335
42,5484097354,5490,64118,99,64858,0,740
44,5484099137,5620,-337,0,0,0,337
49,5484495964,5410,102503,88,102826,0,323
50,5484497951,5420,55376,97,55885,0,509
51,5484504521,5620,72392,88,78230,0,5838


One odd-looking result you might notice in this table: an order that ends up with zero cases filled can still show a shipping cost, and therefore a negative objective. That's not a bug -- it follows directly from the equations doc, which ties shipping cost to *assignment* (which DC the order is assigned to), not to how many cases actually go out the door. In this baseline every order is "assigned" to its default DC by definition, so the lane cost gets charged even when nothing ends up shipping. Across the full dataset this happens to 8 of the 1,109 orders. It's being left as-is here to match the documented formula exactly, but it's worth knowing why it looks strange: in the real world, if nothing ships, no truck goes out and there's nothing to pay for -- the formula just doesn't have a way to capture that.

## Step 14 -- the summary row

Objective value, fill rate, number reassigned, penalty cost, shipping cost -- the five things the to log for the baseline as a whole.

In [22]:
summary = {
    'Objective value [$]': round(baseline_table['Obj. function [$]'].sum()),
    'Fill rate [%]': round(focus_metrics['cases_filled'].sum() / focus_metrics['ordered_qty'].sum() * 100, 1),
    '# reassigned': 0,   # by definition -- nobody moves in this baseline
    'Penalty cost [$]': round(baseline_table['Penalty [$]'].sum()),
    'Shipping cost [$]': round(baseline_table['Shipping cost [$]'].sum())
}

for k, v in summary.items():
    print(f'{k:22s}: {v}')

Objective value [$]   : 45836632
Fill rate [%]         : 93.1
# reassigned          : 0
Penalty cost [$]      : 53206
Shipping cost [$]     : 565479


## Step 15 -- sanity check against the original POC's own numbers

`Output_order_level_data.csv` already has the 2024 POC's own default-scenario columns (`Default_DC_COF`, `DefaultDCShippingCost`, `PenaltyIfNotDiverted`, `DefaultRevenue`). We're not using this file anywhere in the calculation above -- it's purely a check that our independently-built numbers land in the same neighborhood as theirs.

In [23]:
poc_output = pd.read_csv('Output_order_level_data.csv')

# their order id column has a slash in the name, awkward but that's what's in the file
poc_output = poc_output.rename(columns={'SalesDocument/GroupingIndicator': 'Group_Flag'})

In [24]:
compare = order_metrics.merge(
    poc_output[['Group_Flag', 'Default_DC_COF', 'DefaultDCShippingCost', 'PenaltyIfNotDiverted', 'DefaultRevenue']],
    on='Group_Flag', how='inner'
)

compare['fill_rate_diff'] = compare['fill_rate'] - compare['Default_DC_COF']
compare['shipping_diff']  = compare['Shipping_Cost'] - compare['DefaultDCShippingCost']
compare['penalty_diff']   = compare['penalty'] - compare['PenaltyIfNotDiverted']
compare['revenue_diff']   = compare['revenue'] - compare['DefaultRevenue']

print('mean abs fill rate diff:', compare['fill_rate_diff'].abs().mean())
print('mean abs shipping diff :', compare['shipping_diff'].abs().mean())
print('mean abs penalty diff  :', compare['penalty_diff'].abs().mean())
print('mean abs revenue diff  :', compare['revenue_diff'].abs().mean())

mean abs fill rate diff: 0.012668346869216135
mean abs shipping diff : 0.0
mean abs penalty diff  : 53.67856710542203
mean abs revenue diff  : 1668.8330121785384


**Reading these four numbers:**

- **Shipping cost, 0.0** -- exact match, every time. Confirms the plant-to-zip lookup is joining correctly with no errors.
- **Fill rate, ~1.3 points off on average** -- traceable to a real mechanism, not noise. One order, for example, wants 6,930 units and `Available_inventory` that day shows 9,997 -- plenty, so our rule says fully covered. But `TotalDemand` for that same SKU/plant/day is 7,062, not 6,930, meaning other demand we can't see in this order file was also drawing on that same stock. The real POC output only allocated 3,067 of the 6,930 to this order. `Available_inventory` is a shared, once-a-day snapshot split across everything hitting it, not a number reserved just for one order -- our rule checks "is there enough in total," not "does this order actually get served first." By other words we cannot check this now because we need to know which order or part of which order is going to be served first which will be done in for examplr QUBO or greedy algorithms although it violates a restriction.
- **Revenue, ~\$1,700 off on average** -- not a separate error, just the fill-rate gap multiplied by dollar values. A small percentage gap looks bigger once it's converted to dollars on large orders.
- **Penalty, ~\$54 off on average** -- proportionally the largest gap, because the penalty rule is a hard threshold, not a smooth scale. A tiny fill-rate difference can be just enough to flip an order from *no penalty* to *full penalty*.

Overall: the one number that should match exactly does match exactly, and the rest trace back to one already-understood cause. That's a solid validation, not a red flag.

## Notes and what I'd flag in the report

A few honest caveats worth carrying into any write-up of this analysis rather than glossing over:

- The dock-addition proxy flags 25 orders. When I dug into the underlying dock file, only one row in the entire file was a genuinely fully-booked day with real appointments on it (plant 5410, 6/26/24, 35 of 35 slots taken) -- the other zero-remaining rows turn out to be days where the plant had no dock configured at all (weekends, a holiday, a system cutover week), not actual capacity exhaustion. All 25 flagged orders trace back to that one real event, so the count isn't being inflated by those non-operating days -- but it's still standing in for a throughput/case-pick constraint that this data pack doesn't give us a direct way to measure, so treat it as an approximation rather than an exact figure.
- Plants 5083 and 5773 never appear in the dock capacity file at all, so no day for either of them can ever be flagged as constrained by this method -- any orders shipping from those two plants can only ever enter the focus list through the inventory-shortage path, never the dock-addition path.
- 14 order-SKU lines (out of 25,193) had no matching row in the capacity planning file at all -- treated as zero available inventory rather than dropped.
- A handful of `Available_inventory` values in the raw file are slightly negative; floored at 0 before computing cases filled.
- Roughly 1 in 5 order lines have no penalty schedule (`FillRateThreshold`/`Penaltyforpotentialcuts` both null) -- for those orders, penalty is 0 regardless of how badly they missed fill rate, which matches what the original POC output actually does, not just an assumption on our part.
- This baseline never even looks at case-pick, pallet-pick, or dock capacity as *constraints* on the default DC itself -- only as the (approximate) trigger for adding orders to the focus list. That's intentional: the default DC already has these orders in real life, so we're not re-litigating whether it could technically serve them, just whether inventory covers demand.